## AVH data resources

In [3]:
import pandas as pd
import requests

url = 'https://api.ala.org.au/occurrences/occurrences/facets'
params = {
  'q': 'data_hub_uid:dh9',
  'facets': 'data_resource_uid'
}

r = requests.get(url, params=params)

data = r.json()

data_resources = []
for row in data[0]['fieldResult']:
  data_resources.append({
    'uid': row['i18nCode'][row['i18nCode'].find('.') + 1:],
    'label': row['label'],
    'count': row['count']
  })

df_data_resources = pd.DataFrame(data_resources)

## Occurrences with annotations for data resourse

In [4]:
def create_occurrences_df_for_dr(dr):
  url = 'https://api.ala.org.au/occurrences/occurrences/search'
  pageSize = 1000
  start = 0
  totalRecords = 1
  occurrences = []
  while start < totalRecords:
    params = {
      'q': f'data_resource_uid:{dr}',
      'fq': 'user_assertions:*',
      'fl': 'id,data_resource_uid,institutionCode,collectionCode,catalogNumber',
      'pageSize': pageSize,
      'start': start
    }
    r = requests.get(url, params=params)
    data = r.json()
    totalRecords = data['totalRecords']
    occurrences += data['occurrences']
    start += pageSize
  return pd.DataFrame(occurrences).rename(columns={
    'raw_institutionCode': 'institutionCode',
    'raw_collectionCode': 'collectionCode',
    'uuid': 'occurrenceID',
    'raw_catalogNumber': 'catalogNumber'
  })

In [5]:
def create_annotations_df_for_dr(dr):
  df_occurrences = create_occurrences_df_for_dr(dr)
  if df_occurrences.empty:
    return pd.DataFrame([])
  annotations = []
  for index, row in df_occurrences.iterrows():
    uid = row['occurrenceID']
    url = f'https://biocache.ala.org.au/ws/occurrences/{uid}/assertions'
    r = requests.get(url)
    data = r.json()
    annotations += sorted(data, key=lambda item: item['created'])
  df_annotations = pd.DataFrame(annotations)
  return pd.merge(df_occurrences, df_annotations, how="left", left_on="occurrenceID", right_on="referenceRowKey")

In [6]:
for index, row in df_data_resources.iterrows():
  df = create_annotations_df_for_dr(row['uid'])
  if not df.empty:
    df.to_excel('data/' + row['uid'] + '.xlsx', index=False)

In [7]:
museum_drs = ['dr342', 'dr340', 'dr344']
for dr in museum_drs:
  df = create_annotations_df_for_dr(dr)
  if not df.empty:
    df.to_excel('data/' + dr + '.xlsx', index=False)

In [12]:
nrca_drs = ['dr341', 'dr349', 'dr130', 'dr15860', 'dr15867', 'dr15868']
for index, dr in enumerate(nrca_drs):
  if index == 0:
    df = create_annotations_df_for_dr(dr)
  else: 
    df = pd.concat([df, create_annotations_df_for_dr(dr)], ignore_index=True)

  df.rename(columns={
    'dataResourceUid_x': 'dataResourceUid',
  }).drop(columns=['dataResourceUid_y', 'userEmail']).to_excel('data/nrca.xlsx', index=False)

In [8]:
df_df376_occ = create_occurrences_df_for_dr('dr376')
df_df376_occ

,occurrenceID,institutionCode,collectionCode,catalogNumber,dataResourceUid
0,a7c4b60a-2c32-4f69-968f-9d71f8f9de3d,MEL,MEL,MEL 2201036A,dr376
1,147101d1-2896-4281-b694-896340604931,MEL,MEL,MEL 0720450A,dr376
2,cd91e5ed-7c84-4293-aba3-4b9885eb334b,MEL,MEL,MEL 2035934A,dr376
3,f5413753-6193-40fb-b462-683e5b03c962,MEL,MEL,MEL 2122388A,dr376
4,1d8b67b4-a220-41bc-b03f-8c52824247a9,MEL,MEL,MEL 0065430A,dr376
...,...,...,...,...,...
1571,492b21fb-160f-4f8a-84d1-2bbba4978c40,MEL,MEL,MEL 0065226A,dr376
1572,12ad0153-c348-438a-8e38-f9e573c659b8,MEL,MEL,MEL 2222140A,dr376
1573,9118d08c-d3f3-4c87-9271-a0b4c1ff3aa2,MEL,MEL,MEL 0540735A,dr376
1574,9b479ac8-33d4-4fa9-8345-f6ae28b4895d,MEL,MEL,MEL 2069933A,dr376


In [10]:
for index, row in df_df376_occ.iterrows():
  uid = row['occurrenceID']
  url = f'https://biocache.ala.org.au/ws/occurrences/{uid}/assertions'
  r = requests.get(url)
  data = r.json()
  for item in data:
    if 'userEmail' in item:
      print(url)

https://biocache.ala.org.au/ws/occurrences/a7c4b60a-2c32-4f69-968f-9d71f8f9de3d/assertions
https://biocache.ala.org.au/ws/occurrences/f5413753-6193-40fb-b462-683e5b03c962/assertions
https://biocache.ala.org.au/ws/occurrences/1d8b67b4-a220-41bc-b03f-8c52824247a9/assertions
https://biocache.ala.org.au/ws/occurrences/e77a0036-f32c-4782-b0a3-ac5358208604/assertions
https://biocache.ala.org.au/ws/occurrences/28f158b1-08bc-450c-bf22-d448bc6c539a/assertions
https://biocache.ala.org.au/ws/occurrences/28f158b1-08bc-450c-bf22-d448bc6c539a/assertions
https://biocache.ala.org.au/ws/occurrences/1b72f474-ecb5-4ed8-a562-4a788c8eba26/assertions
https://biocache.ala.org.au/ws/occurrences/1b72f474-ecb5-4ed8-a562-4a788c8eba26/assertions
https://biocache.ala.org.au/ws/occurrences/48b33c08-88d0-44e4-8c26-060ac3264045/assertions
https://biocache.ala.org.au/ws/occurrences/c4153169-4d3b-45f8-a5fb-86cc3a6504af/assertions
https://biocache.ala.org.au/ws/occurrences/c4153169-4d3b-45f8-a5fb-86cc3a6504af/assertions